In [1]:
import numpy as np
import scipy.io as sio
from scipy.signal import butter, filtfilt

def bandpass_filter(data, lowcut=4.0, highcut=38.0, fs=250.0, order=5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data, axis=-1)

def exponential_moving_standardize(data, factor_new=0.001, init_block_size=1000, eps=1e-4):
    """
    Apply exponential moving standardization to EEG signal.
    data: shape (channels, time)
    """
    mean = np.mean(data[:, :init_block_size], axis=1, keepdims=True)
    var = np.var(data[:, :init_block_size], axis=1, keepdims=True)
    
    standardized = np.zeros_like(data)
    mu = mean
    sigma = np.sqrt(var + eps)
    for i in range(data.shape[1]):
        if i > 0:
            mu = (1 - factor_new) * mu + factor_new * data[:, i-1:i]
            sigma = (1 - factor_new) * sigma + factor_new * (data[:, i-1:i] - mu)**2
        standardized[:, i] = (data[:, i] - mu.squeeze()) / (np.sqrt(sigma + eps).squeeze())
    return standardized

def load_BCI2a_data(data_path, subject, training=True, all_trials=True):    
    n_channels = 22
    n_tests = 6 * 48     
    window_Length = 7 * 250  # 7s trial

    fs = 250
    t1 = int(2 * fs)  # Start of MI
    t2 = int(6 * fs)  # End of MI

    class_return = np.zeros(n_tests)
    data_return = np.zeros((n_tests, n_channels, window_Length))

    NO_valid_trial = 0
    file_suffix = 'T.mat' if training else 'E.mat'
    a = sio.loadmat(data_path + 'A0' + str(subject + 1) + file_suffix)
    a_data = a['data']

    for ii in range(a_data.size):
        a_data1 = a_data[0, ii]
        a_data2 = [a_data1[0, 0]]
        a_data3 = a_data2[0]
        a_X = a_data3[0]           # EEG data (time x channels)
        a_trial = a_data3[1]       # Trial start indices
        a_y = a_data3[2]           # Class labels
        a_artifacts = a_data3[5]   # Artifact info

        for trial in range(a_trial.size):
            if a_artifacts[trial] != 0 and not all_trials:
                continue

            # Extract and transpose trial to (channels x time)
            trial_data = np.transpose(a_X[int(a_trial[trial]):int(a_trial[trial]) + window_Length, :22])

            # 1. Convert to microvolts
            trial_data = trial_data * 1e6

            # 2. Bandpass filter (4–38 Hz)
            trial_data = bandpass_filter(trial_data, lowcut=4, highcut=38, fs=fs, order=4)

            # 3. Save full 7s trial for now
            data_return[NO_valid_trial, :, :] = trial_data
            class_return[NO_valid_trial] = int(a_y[trial])
            NO_valid_trial += 1        

    # Keep only valid trials and select MI window (2s to 6s)
    data_return = data_return[0:NO_valid_trial, :, t1:t2]
    class_return = class_return[0:NO_valid_trial]
    class_return = (class_return - 1).astype(int)

    # 4. Apply exponential moving standardization to each trial
    data_standardized = np.array([
        exponential_moving_standardize(trial) for trial in data_return
    ])

    return data_standardized, class_return


In [8]:
train_data, labels = load_BCI2a_data('BCI_Kaggle/', subject=3, training=True, all_trials=True)

/var/folders/dh/nfwywq5x6vj6wb3fkdkwkg000000gn/T/ipykernel_3237/3085445733.py:61: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  trial_data = np.transpose(a_X[int(a_trial[trial]):int(a_trial[trial]) + window_Length, :22])
/var/folders/dh/nfwywq5x6vj6wb3fkdkwkg000000gn/T/ipykernel_3237/3085445733.py:71: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  class_return[NO_valid_trial] = int(a_y[trial])


In [9]:
eeg_data = train_data[:, :, 100:400]  # (n_trials, 22, 300)

In [10]:
import numpy as np

def compute_pcc_adjacency(eeg_trials, top_k_ratio=0.2):
    n_trials, n_channels, _ = eeg_trials.shape
    adj_matrices = np.zeros((n_trials, n_channels, n_channels))

    for i in range(n_trials):
        X = eeg_trials[i]  # shape: (22, 300)
        corr = np.corrcoef(X)
        np.fill_diagonal(corr, 0)  # remove self-loops
        
        # Keep top K% strongest absolute correlations
        k = int(top_k_ratio * n_channels * (n_channels - 1) / 2)
        flat = np.abs(corr[np.triu_indices(n_channels, k=1)])
        threshold = np.sort(flat)[-k] if k > 0 else 0
        corr[np.abs(corr) < threshold] = 0

        adj_matrices[i] = corr

    return adj_matrices


def threshold_adjacency(pcc_matrix, threshold_percent=20):
    """
    Thresholds PCC matrix to create adjacency matrix
    Keeps top 'threshold_percent'% of absolute PCC values (excluding diagonal)
    """
    n_channels = pcc_matrix.shape[0]
    flat_pcc = np.abs(pcc_matrix[np.triu_indices(n_channels, k=1)])
    threshold_value = np.percentile(flat_pcc, 100 - threshold_percent)

    adj_matrix = np.where(np.abs(pcc_matrix) >= threshold_value, 1, 0)
    np.fill_diagonal(adj_matrix, 0)  # Remove self-loops if needed
    return adj_matrix


In [11]:
adj_matrices = compute_pcc_adjacency(eeg_data, top_k_ratio=0.2)

In [12]:
import tensorflow as tf

X_graph = tf.convert_to_tensor(eeg_data, dtype=tf.float32)        # (n_samples, 22, 300)
X_adj = tf.convert_to_tensor(adj_matrices, dtype=tf.float32)      # (n_samples, 22, 22)
y_labels = tf.convert_to_tensor(labels, dtype=tf.float32)         # (n_samples,)

In [5]:
# def construct_graphs(eeg_data):
#     """
#     eeg_data: shape (trials, 22, 1000)
#     Returns:
#         graphs: list of dicts with keys 'adj' (adjacency), 'features' (node features)
#     """
#     graphs = []
#     for trial in eeg_data:
#         pcc = compute_pcc_matrix(trial)
#         adj = threshold_adjacency(pcc, threshold_percent=20)
#         graphs.append({
#             'adj': adj.astype(np.float32),           # (22, 22)
#             'features': trial.T.astype(np.float32)   # (1000, 22) or keep (22, 1000) depending on model
#         })
#     return graphs

In [13]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Custom Graph Convolution Layer
class GraphConv(tf.keras.layers.Layer):
    def __init__(self, units, activation=None):
        super(GraphConv, self).__init__()
        self.units = units
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        feature_shape = input_shape[0][-1]
        self.w = self.add_weight(shape=(feature_shape, self.units),
                                 initializer='glorot_uniform',
                                 trainable=True)

    def call(self, inputs):
        x, adj = inputs  # x: (batch, nodes, features), adj: (batch, nodes, nodes)
        xw = tf.matmul(x, self.w)
        out = tf.matmul(adj, xw)  # Graph convolution: A * XW
        return self.activation(out)

In [14]:
def forward_forward_loss(y_true, y_pred):
    # Assumes y_true shape: (batch, ), binary labels (0 or 1)
    y_true = tf.cast(tf.squeeze(y_true), tf.bool)
    
    positive = tf.boolean_mask(y_pred, y_true)       # Predicted features for class 1
    negative = tf.boolean_mask(y_pred, ~y_true)      # Predicted features for class 0

    # FF loss: encourage activations for positive and suppress for negative
    pos_loss = tf.reduce_sum(tf.square(positive))            # encourage strong signal
    neg_loss = tf.reduce_sum(tf.square(1.0 - negative))      # suppress weak signal
    return pos_loss + neg_loss


In [15]:
def build_ffgcn_model(num_nodes=22, num_timesteps=300, output_dim=32):
    graph_input = Input(shape=(num_nodes, num_timesteps), name='graph_input')
    adjacency_input = Input(shape=(num_nodes, num_nodes), name='adjacency_input')

    # Optional preprocessing (e.g., linear projection)
    x = GraphConv(64, activation='relu')([graph_input, adjacency_input])
    x = GraphConv(output_dim, activation='relu')([x, adjacency_input])

    # Pool across nodes or timesteps to get graph-level representation
    x = GlobalAveragePooling1D()(x)  # shape (batch, output_dim)

    model = Model(inputs=[graph_input, adjacency_input], outputs=x)
    model.compile(optimizer=Adam(1e-3), loss=forward_forward_loss)
    return model


In [17]:
from sklearn.model_selection import train_test_split
import tensorflow as tf
import numpy as np

# Assuming eeg_data, adj_matrices, labels are already prepared
# Shape: eeg_data (samples, 22, 300), adj_matrices (samples, 22, 22), labels (samples,)

# Step 1: Train-Test Split
X_train_graph, X_test_graph, X_train_adj, X_test_adj, y_train, y_test = train_test_split(
    eeg_data, adj_matrices, labels, test_size=0.2, random_state=42, stratify=labels
)

# Convert to tensors
X_train_graph = tf.convert_to_tensor(X_train_graph, dtype=tf.float32)
X_train_adj = tf.convert_to_tensor(X_train_adj, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train, dtype=tf.int32)

X_test_graph = tf.convert_to_tensor(X_test_graph, dtype=tf.float32)
X_test_adj = tf.convert_to_tensor(X_test_adj, dtype=tf.float32)
y_test = tf.convert_to_tensor(y_test, dtype=tf.int32)

# Step 2: Build classification model
def build_classification_model(num_nodes=22, num_timesteps=300, output_dim=32, num_classes=4):
    graph_input = tf.keras.Input(shape=(num_nodes, num_timesteps))
    adj_input = tf.keras.Input(shape=(num_nodes, num_nodes))

    x = GraphConv(64, activation='relu')([graph_input, adj_input])
    x = GraphConv(32, activation='relu')([x, adj_input])

    x = tf.keras.layers.GlobalAveragePooling1D()(x)  # flatten node features
    out = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs=[graph_input, adj_input], outputs=out)
    return model

model = build_classification_model()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Step 3: Train and print accuracies after each epoch
class AccuracyLogger(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        train_loss, train_acc = self.model.evaluate(
            [X_train_graph, X_train_adj], y_train, verbose=0)
        test_loss, test_acc = self.model.evaluate(
            [X_test_graph, X_test_adj], y_test, verbose=0)
        print(f"Epoch {epoch + 1}: Train Acc = {train_acc:.4f}, Test Acc = {test_acc:.4f}")

# Step 4: Fit the model
model.fit([X_train_graph, X_train_adj], y_train,
          validation_data=([X_test_graph, X_test_adj], y_test),
          epochs=20,
          batch_size=32,
          callbacks=[AccuracyLogger()])

Epoch 1/20
6/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2619 - loss: 44.3324 Epoch 1: Train Acc = 0.3478, Test Acc = 0.2759
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 94ms/step - accuracy: 0.2521 - loss: 42.2232 - val_accuracy: 0.2759 - val_loss: 19.8216
Epoch 2/20
1/8 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4688 - loss: 11.2308Epoch 2: Train Acc = 0.4957, Test Acc = 0.3621
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.4103 - loss: 14.6244 - val_accuracy: 0.3621 - val_loss: 14.7324
Epoch 3/20
1/8 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.4375 - loss: 12.9923Epoch 3: Train Acc = 0.6435, Test Acc = 0.3276
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.4929 - loss: 10.1666 - val_accuracy: 0.3276 - val_loss: 13.5650
Epoch 4/20
1/8 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.5625 - loss: 7.1999Epoch 4: Train Acc = 0.7435, Test Acc = 0.2931
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6408 - loss: 4.7245 - val_accuracy: 0.2931 - val_loss: 12.6841
Epoch 5/20
1/8 ━━